In [ ]:
import json
import uuid
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams, QueryResponse

### Read the sampled dataset with Amazon inventory data

In [ ]:
DATA_DIR = Path("../../data")
ITEMS_PATH = DATA_DIR / "meta_Electronics_2022_2023_with_category_ratings_100_sample_1000.jsonl"

In [ ]:
df_items = pd.read_json(ITEMS_PATH, lines=True)

In [ ]:
df_items.head()

In [ ]:
df_items["features"].head(5)

### Preprocess title and features

In [ ]:
def preprocess_description(row) -> str:
    title = row.get("title") or ""
    features = row.get("features") or []
    return " ".join(f"{title} {' '.join(features)}".split())


def extract_first_large_image(row) -> str | None:
    images = row.get("images") or []
    return images[0].get("large") if images else None


df_items["preprocess_description"] = df_items.apply(preprocess_description, axis=1)
df_items["image"] = df_items.apply(extract_first_large_image, axis=1)

In [ ]:
df_items.head()

In [ ]:
df_items["preprocess_description"].iloc[0]

In [ ]:
df_items["image"].iloc[0]

### Sample 50 items from the dataset

In [ ]:
df_sample = df_items.sample(n=50, random_state=42)
len(df_sample)

In [ ]:
df_data_to_embed = df_sample[
    ["preprocess_description", "image", "rating_number", "price", "average_rating", "parent_asin"]
]
df_data_to_embed.head()

In [ ]:
data_to_embed = json.loads(df_data_to_embed.to_json(orient="records"))
data_to_embed[0]

### Define the embedding function

In [ ]:
COLLECTION_NAME = "Amazon-items-collection-01"
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_DIM = 1536

load_dotenv("../../.env")
client = OpenAI()

In [ ]:
response = client.embeddings.create(
    input="Random text",
    model=EMBEDDING_MODEL,
)

len(response.data[0].embedding)

In [ ]:
response.data[0].embedding

In [ ]:
def get_embeddings(
    texts: list[str],
    model: str = EMBEDDING_MODEL,
    batch_size: int = 100,
) -> list[list[float]]:
    embeddings: list[list[float]] = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(input=batch, model=model)
        embeddings.extend(d.embedding for d in sorted(resp.data, key=lambda d: d.index))
    return embeddings


def get_embedding(text: str, model: str = EMBEDDING_MODEL) -> list[float]:
    return get_embeddings([text], model=model)[0]


In [ ]:
len(get_embedding("Hi."))

### Create Qdrant Collection

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

if not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
    )

### Embed Data

#### PointStruct example

In [ ]:
pointstruct = PointStruct(
    id=0,
    vector=get_embedding("Hello, world!"),
    payload={
        "text": "Hello, world!",
        "model": EMBEDDING_MODEL,
    },
)
pointstruct

In [ ]:
len(pointstruct.vector)

#### Amazon Data

In [ ]:
texts = [item["preprocess_description"] for item in data_to_embed]
embeddings = get_embeddings(texts, model=EMBEDDING_MODEL, batch_size=100)

pointstructs = [
    PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_DNS, item["parent_asin"])),
        vector=embedding,
        payload=item,
    )
    for item, embedding in zip(data_to_embed, embeddings)
]
len(pointstructs)

In [ ]:
len(embeddings[0])

### Write embedded data to Qdrant

In [ ]:
qdrant_client.upload_points(
    collection_name=COLLECTION_NAME,
    points=pointstructs,
    batch_size=100,
)
qdrant_client.count(COLLECTION_NAME)

### Define a function for data retrieval

In [ ]:
def retrieve_data(query: str, k: int = 5) -> QueryResponse:
    query_embedding = get_embedding(query)
    return qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=k,
    )

In [ ]:
response = retrieve_data("Do you have a USB connectable fan for hot summers.", k=10)
response.points

In [ ]:
for point in retrieve_data("Do you have a USB connectable fan for hot summers.", k=10).points:
    print(point.payload["parent_asin"], point.id)